# SO(3) Riemannian Flow Matching (`rfm_so3`)

`rfm_so3` keeps everything of `fm_rel_pose` (relative frame, pose-level output, conditioning, network family, data, optimiser, 20k steps, seed) and changes the generative process for rotations: Haar-uniform prior on SO(3), geodesic interpolation, body angular-velocity targets, and a sampler that stays on SO(3) (`R ← R·Exp(dt·ω)`). Positions keep Euclidean FM. This is a **method-level** comparison (prior, state parameterisation and integrator all differ), on the product space SO(3)ᴸ × ℝ³ᴸ, not a coupled SE(3) flow.

Main question (H3): does it change orientation fidelity on **E1b** (excursion W1, out-of-support rate)? E1a mode/endpoint metrics are secondary. Everything is saved to Google Drive. ≈ 40–55 min on a T4.

## 1. Code + Drive

In [ ]:
import os, sys, glob, json, subprocess
REPO_URL = ""  # Optional Git remote URL for Colab; otherwise upload the repository ZIP.
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("camera_motion"):
    candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if not candidates and IN_COLAB:
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "camera-motion-flow-matching"], check=True)
        else:
            from google.colab import files
            from zipfile import ZipFile
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Upload the repository ZIP or specify REPO_URL.")
            with ZipFile(next(iter(uploaded))) as zf:
                zf.extractall(".")
        candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if candidates:
        os.chdir(candidates[0])
assert os.path.isdir("camera_motion"), f"Cannot find repository root: {os.getcwd()}"
sys.path.insert(0, os.getcwd())
print("repository root:", os.getcwd())


RUNS = "runs"
if IN_COLAB:                                       # default: everything goes to Drive
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/camera_motion_runs"
os.makedirs(RUNS, exist_ok=True)
SEED = 0
FAM = ("E1a", "E1b")
POSE = {f: f"{RUNS}/fm_rel_pose_{f}_s{SEED}" for f in FAM}
INC  = {f: f"{RUNS}/fm_rel_inc_{f}_s{SEED}" for f in FAM}
RFM  = {f: f"{RUNS}/rfm_so3_{f}_s{SEED}" for f in FAM}
EV = "eval_m2_K200_nfe50_euler"
for name, d in (("rel_pose", POSE), ("rel_inc", INC)):
    for f in FAM:
        print(f"{name:8s} {f} eval:", "OK" if os.path.exists(f"{d[f]}/{EV}/eval_report.json") else "missing (comparison rows will be skipped)")
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-600:]); assert r.returncode == 0, r.stderr[-1500:]

## 3. Train `rfm_so3` (identical budget). Re-running resumes from Drive.

In [ ]:
!python -u scripts/train_fm.py --rep rfm_so3 --family E1a --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 2000 --out {RFM['E1a']}

In [ ]:
!python -u scripts/train_fm.py --rep rfm_so3 --family E1b --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 2000 --out {RFM['E1b']}

## 4. Evaluate + solver sweep
The E1b sweep is run for both `rfm_so3` and `fm_rel_pose` (if its checkpoint is on Drive), so orientation metrics can be compared at matched solver settings.

In [ ]:
for f in FAM:
    !python -u scripts/eval_fm.py --run {RFM[f]} --K 200 --nfe 50
GRID = "euler:20 euler:50 euler:100 midpoint:50 midpoint:100"
!python -u scripts/diagnose_solver.py --run {RFM['E1a']} --panel eval_id --grid {GRID}
!python -u scripts/diagnose_solver.py --run {RFM['E1b']} --panel eval_id --grid {GRID}
if os.path.exists(f"{POSE['E1b']}/ckpt_last.pt"):
    !python -u scripts/diagnose_solver.py --run {POSE['E1b']} --panel eval_id --grid {GRID}

## 5. Matched comparison: a = `fm_rel_pose`, b = `rfm_so3`

In [ ]:
CMP = f"{RUNS}/comparisons/gate5_rel_pose_vs_rfm_so3"
for f in FAM:
    if os.path.exists(f"{POSE[f]}/{EV}/eval_report.json"):
        !python -u scripts/matched_comparison.py --family {f} --a {POSE[f]}/{EV} --b {RFM[f]}/{EV} --out {CMP} --label-a rel_pose --label-b rfm_so3
        !python -u scripts/compare_runs.py --a {POSE[f]}/{EV} --b {RFM[f]}/{EV}

## 6. Demo videos + three-model benchmark

In [ ]:
DEMO, BENCH = f"{RUNS}/demo", f"{RUNS}/benchmark"
!python -u scripts/render_demo.py modes --eval-dir {RFM['E1a']}/{EV} --family E1a --out {DEMO}
!python -u scripts/render_demo.py gt_vs_gen --eval-dir {RFM['E1b']}/{EV} --family E1b --out {DEMO}
runs = []
for name, d in (("rel_pose", POSE), ("rel_inc", INC), ("rfm_so3", RFM)):
    for f in FAM:
        if os.path.exists(f"{d[f]}/{EV}/eval_report.json"):
            runs += ["--run", f"{name}={d[f]}/{EV}"]
subprocess.run([sys.executable, "scripts/make_benchmark.py", *runs, "--out", BENCH, "--condition", "1"], check=True)
from IPython.display import Image, Video, Markdown, display
display(Markdown(open(f"{BENCH}/benchmark_table.md").read()))
for f in FAM:
    p = f"{CMP}/{f}_matched_comparison.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for p in sorted(glob.glob(f"{BENCH}/*.png")) + sorted(glob.glob(f"{RFM['E1b']}/{EV}/figures/*.png")):
    print(p); display(Image(filename=p))
for v in sorted(glob.glob(f"{DEMO}/rfm_so3*.mp4")):
    print(v); display(Video(v, embed=True, width=900))

## 7. Manifest + pack (no checkpoints)

In [ ]:
!python -u scripts/write_manifest.py --runs {RUNS} --out {RUNS}/experiment_manifest.json
import zipfile
with zipfile.ZipFile("gate5_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{RUNS}/experiment_manifest.json", "experiment_manifest.json")
    dirs = list(RFM.values()) + [f"{POSE['E1b']}/diagnostics", CMP, BENCH]
    for d in dirs:
        for f in glob.glob(f"{d}/**/*", recursive=True):
            if os.path.isdir(f) or f.endswith(".pt"):
                continue
            z.write(f, os.path.relpath(f, RUNS))
    for f in glob.glob(f"{DEMO}/rfm_so3*"):
        z.write(f, os.path.relpath(f, RUNS))
print("gate5_results.zip", round(os.path.getsize("gate5_results.zip") / 2**20, 1), "MB")
if IN_COLAB:
    from google.colab import files; files.download("gate5_results.zip")